# Structural Alert Performance Analysis (Benigni-Bossa Alerts)

This notebook replicates the **Table 7** style analysis from the *AmesFormer* paper. It maps test predictions to standard **Benigni-Bossa mutagenicity alerts** and computes standard toxicological counts (TP, FP, TN, FN) and performance metrics (Sensitivity, Specificity, Accuracy) inside each alert category.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from tox_ames.utils.structural_alerts import analyze_structural_alerts


## 1. List Available Models
We list the available CV runs in your log directories.

In [ ]:
logs_dir = Path("logs/combined")
available_models = sorted(
    [
        d.name
        for d in logs_dir.iterdir()
        if d.is_dir() and (d / "cv_predictions.csv").exists()
    ]
)

print("Available Models:")
for idx, name in enumerate(available_models, start=1):
    print(f"{idx:2d}. {name}")


## 2. Evaluate Predictions Against Structural Alerts
Choose a model folder name from the list above and run the cells to calculate and display the metrics table.

In [ ]:
selected_model = "exp-unimolv2_ft-scf"
df_alerts = analyze_structural_alerts(selected_model)


## 3. Results Table
We sort the structural alerts by prevalence (count of matching molecules in the test set) and display the performance indices.

In [ ]:
# Filter out alerts that have 0 occurrences in the dataset
df_present = df_alerts[df_alerts["Total Count"] > 0].copy()
df_sorted = df_present.sort_values(by="Total Count", ascending=False).reset_index(
    drop=True
)

# Format metric percentages for readability
df_display = df_sorted.copy()
df_display["Sensitivity"] = df_display["Sensitivity"].map(
    lambda x: f"{x:.2%}" if not pd.isna(x) else "N/A"
)
df_display["Specificity"] = df_display["Specificity"].map(
    lambda x: f"{x:.2%}" if not pd.isna(x) else "N/A"
)
df_display["Accuracy"] = df_display["Accuracy"].map(
    lambda x: f"{x:.2%}" if not pd.isna(x) else "N/A"
)

df_display


## 4. Performance Plot

In [ ]:
from tox_ames.utils.structural_alerts import plot_structural_alerts

plot_structural_alerts(
    df_alerts,
    save_path=Path("Figs/structural_alerts_performance.png"),
    model_name=selected_model,
)


## 5. Structural-alert performance by Ames outcome
For fine-tuned Uni-Mol v2, we compare sensitivity among Ames-positive compounds with specificity among Ames-negative compounds for the same well-supported Benigni--Bossa alert classes. The figure reports mean $\pm$ sample standard deviation across repeated holdouts and unique-compound support.

In [ ]:
from IPython.display import Image, display
from pathlib import Path
from tox_ames.utils.structural_alerts import plot_alert_class_performance

# Re-run this cell to replace the legacy cached output below.
png_path = Path("Figs/structural_alerts_unimolv2.png")
alert_summary = plot_alert_class_performance(selected_model, save_path=png_path)
display(alert_summary)
display(Image(filename=png_path))


## 6. Cross-representation ensemble
The ensemble is scored through the same function and the same alert definitions as the single-pipeline analysis above, so the two columns of the supporting-information table are computed identically. Predictions are read from `advanced_analysis/ensemble_predictions.csv`.

In [ ]:
ensemble_summary = plot_alert_class_performance(
    "advanced_analysis",
    save_path=Path("Figs/structural_alerts_ensemble.png"),
    pred_file="ensemble_predictions.csv",
)
display(ensemble_summary)
display(Image(filename="Figs/structural_alerts_ensemble.png"))

In [ ]:
from IPython.display import display, HTML, Image
from pathlib import Path
from tox_ames.utils.structural_alerts import (
    generate_structural_alerts_table_html,
    generate_structural_alerts_table_png,
)

# 1. Display styled HTML table
html_table = generate_structural_alerts_table_html(selected_model, top_n=20)
display(HTML(html_table))

# 2. Render and display high-res PNG table
png_path = Path("Figs/structural_alerts_table.png")
generate_structural_alerts_table_png(selected_model, top_n=5, save_path=png_path)
display(Image(filename=png_path))
